# Prompt Chaining

Break a complex task into ordered model calls where each output becomes the next step's input.



## Core ideas



- Decomposition: split a large task into smaller operations with clear inputs and outputs.
- Intermediate artifacts: summaries, extracted fields, plans, critiques, or transformed data passed between steps.
- Validation gates: check each step before downstream work amplifies mistakes.
- Context control: each prompt should receive only the information needed for that step.
- Pipeline design: linear chains are simple, while branching chains need routing and error recovery.


## Common failure modes

- Making every workflow a chain even when one well-scoped prompt is enough.
- Passing large unfiltered outputs forward and recreating context overload.
- Skipping intermediate validation.


## Framework implementation

Start with ordinary LangChain calls. Each step has a narrow prompt and passes a small artifact forward; a graph would add no value to this fixed linear flow.


In [ ]:
# `os` is part of Python's standard library. We use it to read environment variables.
import os

# Import two LangChain building blocks:
# - `init_chat_model` creates a chat-model object.
# - `ChatPromptTemplate` creates reusable messages with placeholders.
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

# API keys should be stored outside the notebook. `getenv` returns the key when
# present and returns `None` when it is missing.
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    # This branch runs when no key is configured, avoiding a failed paid request.
    print("Skipped: set OPENAI_API_KEY to run the LangChain chain.")
else:
    # Construct the model client. No API request happens on this line.
    # Temperature 0 asks for conservative sampling but does not guarantee correctness.
    model = init_chat_model("openai:gpt-5.6-sol", temperature=0)

    # A prompt is a list of role/content tuples. `{request}` is a placeholder.
    extract_prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract audience, task, and output format. Return concise JSON."),
        ("user", "{request}"),
    ])

    # LangChain overloads `|` to connect runnable components. The result accepts
    # a dictionary, formats the prompt, calls the model, and returns an AIMessage.
    extract_chain = extract_prompt | model

    draft_prompt = ChatPromptTemplate.from_messages([
        ("system", "Draft a useful answer from these requirements."),
        ("user", "{requirements}"),
    ])
    draft_chain = draft_prompt | model

    polish_prompt = ChatPromptTemplate.from_messages([
        ("system", "Make this concise and actionable without adding unsupported facts."),
        ("user", "{draft}"),
    ])
    polish_chain = polish_prompt | model

    # First API call: replace `{request}` and extract a requirements artifact.
    requirements_message = extract_chain.invoke({
        "request": "Teach prompt chaining to Python developers."
    })
    requirements = requirements_message.content
    print("REQUIREMENTS\n", requirements)

    # Second API call: the previous text becomes this step's input.
    draft_message = draft_chain.invoke({"requirements": requirements})
    first_draft = draft_message.content
    print("\nFIRST DRAFT\n", first_draft)

    # Third API call: polish the draft and print the final text.
    final_message = polish_chain.invoke({"draft": first_draft})
    final = final_message.content
    print("\nFINAL\n", final)


## Offline mechanics

This version runs without credentials and exposes the control flow directly.


In [ ]:
# Define the data shape and small operations before running them.
def extract_requirements(request):
    return {"audience": "developers", "task": request, "format": "checklist"}

def draft(requirements):
    return [f"Explain {requirements['task']}", "Show one concrete example", "Add a practice task"]

def polish(items):
    return "\n".join(f"- {item}" for item in items)

request = "prompt chaining for customer-support analysis"
polish(draft(extract_requirements(request)))
